In [8]:
import mlflow

mlflow.set_experiment("Deep Learning Experiment")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1794022052077, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1794022052077, lifecycle_stage='active', name='Deep Learning Experiment', tags={}, trace_location=None, workspace='default'>

In [9]:
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST("data", train=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

In [11]:
device

device(type='cuda')

In [12]:
import torch.nn as nn


class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)

In [13]:
# Training parameters
params = {
    "epochs": 5,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "optimizer": "SGD",
    "model_type": "MLP",
    "hidden_units": [512, 512],
}

# Define optimizer and loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"])

In [14]:
import copy

# pt2 serialization traces the model, so MLflow needs an example input (batch > 1 keeps the batch dim dynamic)
input_example = next(iter(train_loader))[0][:2].numpy()


def log_cpu_model(model, name):
    # MLflow infers the signature on CPU, so log a CPU copy and leave the training model on the GPU
    return mlflow.pytorch.log_model(copy.deepcopy(model).cpu(), name=name, input_example=input_example)


with mlflow.start_run() as run:
    # Log training parameters
    mlflow.log_params(params)

    for epoch in range(params["epochs"]):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Forward pass
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Calculate metrics
            train_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Log batch metrics (every 100 batches)
            if batch_idx % 100 == 0:
                batch_loss = train_loss / (batch_idx + 1)
                batch_acc = 100.0 * correct / total
                mlflow.log_metrics(
                    {"batch_loss": batch_loss, "batch_accuracy": batch_acc},
                    step=epoch * len(train_loader) + batch_idx,
                )

        # Calculate epoch metrics
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = loss_fn(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        # Calculate and log epoch validation metrics
        val_loss = val_loss / len(test_loader)
        val_acc = 100.0 * val_correct / val_total

        # Log epoch metrics
        mlflow.log_metrics(
            {
                "train_loss": epoch_loss,
                "train_accuracy": epoch_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            },
            step=epoch,
        )
        # Log checkpoint at the end of each epoch
        log_cpu_model(model, f"checkpoint_{epoch}")

        print(
            f"Epoch {epoch + 1}/{params['epochs']}, "
            f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
        )

    # Log the final trained model
    model_info = log_cpu_model(model, "final_model")

2026/11/07 12:26:53 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/07 12:26:53 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/07 12:27:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpdeb2mlyw/model/data, flavor: pytorch). Fall back to return ['torch==2.15.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/11/07 12:27:43 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.predict(model_uri=..., input_data=serving_input, env_manager="uv")` on the model uri and serving input example. A ser

Epoch 1/5, Train Loss: 1.7437, Train Acc: 55.67%, Val Loss: 1.2484, Val Acc: 64.60%


2026/11/07 12:28:11 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpjujhgwu2/model/data, flavor: pytorch). Fall back to return ['torch==2.15.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/11/07 12:28:11 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.predict(model_uri=..., input_data=serving_input, env_manager="uv")` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: The saved model is exported by `torch.export` API, the original model contains weights / buffers on 'cpu

Epoch 2/5, Train Loss: 1.0244, Train Acc: 67.97%, Val Loss: 0.8952, Val Acc: 69.74%


2026/11/07 12:28:36 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpvfigmbnx/model/data, flavor: pytorch). Fall back to return ['torch==2.15.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/11/07 12:28:36 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.predict(model_uri=..., input_data=serving_input, env_manager="uv")` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: The saved model is exported by `torch.export` API, the original model contains weights / buffers on 'cpu

Epoch 3/5, Train Loss: 0.8157, Train Acc: 72.85%, Val Loss: 0.7712, Val Acc: 74.18%


... (2 outputs left out)

2026/11/07 12:29:24 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpvd_vj3d3/model/data, flavor: pytorch). Fall back to return ['torch==2.15.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/11/07 12:29:24 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.predict(model_uri=..., input_data=serving_input, env_manager="uv")` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: The saved model is exported by `torch.export` API, the original model contains weights / buffers on 'cpu

Epoch 5/5, Train Loss: 0.6577, Train Acc: 78.16%, Val Loss: 0.6459, Val Acc: 77.76%


2026/11/07 12:29:30 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp952ntn6w/model/data, flavor: pytorch). Fall back to return ['torch==2.15.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/11/07 12:29:30 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.predict(model_uri=..., input_data=serving_input, env_manager="uv")` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: The saved model is exported by `torch.export` API, the original model contains weights / buffers on 'cpu

🏃 View run nimble-squirrel-904 at: http://localhost:5000/#/experiments/4/runs/189c0545a2f94c7685689a179e886927
🧪 View experiment at: http://localhost:5000/#/experiments/4


In [15]:
run_id = "189c0545a2f94c7685689a179e886927"

model = mlflow.pytorch.load_model(f"runs:/{run_id}/final_model")

model.to(device)
model.eval()

with mlflow.start_run(run_id=run.info.run_id) as run:
    
    test_loss, test_correct, test_total = 0, 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = loss_fn(output, target)
            
            test_loss += loss.item()
            _, predicted = output.max(1)
            test_total += target.size(0)
            test_correct += predicted.eq(target).sum().item()
            
    test_loss = test_loss / len(test_loader)
    test_acc = 100.0 * test_correct / test_total
    
    mlflow.log_metrics({
        "test_loss": test_loss, "test_accuracy": test_acc 
    })
    print(f"Final Test Accuracy: {test_acc:.2f}%")


NotImplementedError: Calling eval() is not supported yet.